# SpaceX Falcon 9 - EDA with SQL

Running the same kind of exploratory queries the course asks for, using SQLite loaded from the SpaceX launch records CSV (this stands in for Db2 - the SQL itself is the same, just running against a local SQLite file instead of a hosted Db2 instance).

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')
df = pd.read_csv('Spacex.csv')
df.to_sql('SPACEXTBL', conn, if_exists='replace', index=False)

cur = conn.cursor()

def run_query(sql):
    return pd.read_sql_query(sql, conn)

df.head()

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing _Outcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 1: Display the names of the unique launch sites

In [2]:
run_query('SELECT DISTINCT Launch_Site FROM SPACEXTBL')

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Task 2: Display 5 records where launch sites begin with 'CCA'

In [3]:
run_query("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing _Outcome
0,2010-04-06,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-08-12,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-08-10,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-01-03,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3: Total payload mass carried by boosters launched for NASA (CRS)

In [4]:
run_query("SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_Mass FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'")

,Total_Payload_Mass
0,45596


### Task 4: Average payload mass carried by booster version F9 v1.1

In [5]:
run_query("SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass FROM SPACEXTBL WHERE Booster_Version LIKE '%F9 v1.1%'")

,Avg_Payload_Mass
0,2534.666667


### Task 5: Date of the first successful landing outcome on a ground pad

In [6]:
run_query("SELECT MIN(Date) AS First_Successful_Ground_Pad_Landing FROM SPACEXTBL WHERE \"Landing _Outcome\" = 'Success (ground pad)'")

,First_Successful_Ground_Pad_Landing
0,2015-12-22


### Task 6: Names of boosters that succeeded on a drone ship with payload mass between 4000 and 6000 kg

In [7]:
run_query("SELECT Booster_Version FROM SPACEXTBL WHERE \"Landing _Outcome\" = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### Task 7: Total number of successful and failure mission outcomes

In [8]:
run_query('SELECT Mission_Outcome, COUNT(*) AS Count FROM SPACEXTBL GROUP BY Mission_Outcome')

,Mission_Outcome,Count
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


### Task 8: Booster version(s) that carried the maximum payload mass

In [9]:
run_query('SELECT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)')

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### Task 9: 2015 records with failure landing outcomes on a drone ship, with month names

In [10]:
run_query("SELECT CASE substr(Date,6,2) WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March' WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June' WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September' WHEN '10' THEN 'October' WHEN '11' THEN 'November' WHEN '12' THEN 'December' END AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM (SELECT Date, \"Landing _Outcome\" AS Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL) WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date,1,4) = '2015'")

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,October,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,April,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10: Rank landing outcomes between 2010-06-04 and 2017-03-20, most common first

In [11]:
run_query("SELECT \"Landing _Outcome\", COUNT(*) AS Outcome_Count FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY \"Landing _Outcome\" ORDER BY Outcome_Count DESC")

,Landing _Outcome,Outcome_Count
0,No attempt,10
1,Success (drone ship),6
2,Success (ground pad),5
3,Failure (drone ship),5
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Precluded (drone ship),1
7,Failure (parachute),1


## Summary

NASA (CRS) missions account for a meaningful chunk of total payload mass across the program, F9 v1.1 averaged a modest payload compared to later boosters, and drone-ship landings show up far more often as the dataset moves into 2015-2017 - reflecting SpaceX's shift toward attempting (and increasingly succeeding at) ocean landings during that window.